# Margen ±: el error base

El margen de la app junta tres errores: `± = √(error base² + Routine que falta² + Extras que faltan²)`. Todos los mide `modeling/s7_margin.py` al entrenar; aquí se ve de dónde sale el error base.

- **Error base:** la calibración en años. En cada décimo de riesgo, cuántos años habría que mover la edad biológica para que el riesgo predicho cuadre con las muertes de verdad.
- **Lo que falta:** la mediana del cambio de la edad según cuántos valores faltan (`missing_values.ipynb`).

Cada persona se predice con el modelo del bloque que no la vio (los 5 bloques de train); el test no se toca. Las predicciones de los 5 bloques **se juntan antes de hacer los décimos**: con cada bloque por separado, los décimos de riesgo bajo tenían tan pocas muertes que el error era sobre todo ruido. El error base de la app es **una sola curva, la de `routine`** (el nivel con más muertes): el error sin signo de cada décimo, obligado a no subir cuando sube el riesgo (regresión isotónica), y unido por rectas.

In [1]:
# Imports
import numpy as np
import pandas as pd

from bioage.config.model_levels import LEVEL_COLUMNS
from bioage.config.paths import GOLD_TABLE
from bioage.config.study import HORIZON_MONTHS
from bioage.modeling.s1_split import split_data
from bioage.modeling.s4_biological_age import biological_age, fit_reference
from bioage.modeling.s5_evaluation import calibration_in_years, observed_risk, years_to_observed
from bioage.modeling.s7_margin import base_margin, fit_blocks, predict_blocks

C:\Data\1_MisCosas\Evolve\TFM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
gold = pd.read_parquet(GOLD_TABLE)
train, test, folds = split_data(gold)
blocks = {level: fit_blocks(train, folds, columns) for level, columns in LEVEL_COLUMNS.items()}

## 1. Calibración en años, con los bloques juntos

In [3]:
def deaths(data: pd.DataFrame) -> int:
    """Deaths within the horizon."""
    return int(((data["death"] == 1) & (data["follow_up_months"] <= HORIZON_MONTHS)).sum())


tables, ages = {}, {}
for level in LEVEL_COLUMNS:
    risk = predict_blocks(blocks[level], train)["risk"]
    reference = fit_reference(risk, train)
    ages[level] = (biological_age(risk, train["sex"], reference), reference)
    table = calibration_in_years(risk, ages[level][0], train, reference)
    table["deaths"] = train.groupby(pd.qcut(risk, len(table), labels=False)).apply(deaths, include_groups=False)
    tables[level] = table
pd.concat({level: table[["deaths", "age", "years"]] for level, table in tables.items()}, axis=1).round(1)

routine             extras             questionnaire               full  \
      deaths   age years deaths   age years        deaths   age years deaths   
risk                                                                           
0         19  16.7   3.4     19  17.0   4.8            20  15.9   8.4     17   
1         29  24.6   2.8     28  25.1   4.6            28  24.8   5.8     32   
2         56  30.9   6.5     46  31.5   6.1            43  31.6   4.6     32   
3         43  37.2  -3.6     46  37.7  -0.5            41  37.9  -3.1     42   
4         84  43.3  -0.3     65  44.0  -2.3            81  44.1   0.2     72   
5        121  49.2  -1.5    106  50.1  -2.1           105  50.2  -2.7    100   
6        206  55.1  -0.2    163  56.2  -2.1           181  56.2  -1.4    156   
7        332  61.2  -0.0    315  62.4  -0.2           328  62.2  -0.4    300   
8        614  67.9   0.2    591  69.1   0.4           628  68.8   0.4    600   
9       1349  77.2   0.0   1474  79.5   0.4          1398  78.4   0.2   1502   

                  
       age years  
risk              
0     16.0   7.0  
1     25.0   7.9  
2     31.6   2.6  
3     38.0  -0.8  
4     44.3  -0.5  
5     50.5  -2.2  
6     56.7  -2.4  
7     62.8  -0.6  
8     69.7   0.2  
9     80.2   0.6

## 2. El error base: la curva de `routine`

Cada punto es un décimo de riesgo: su riesgo medio y su error en años. La isotónica es el cambio mínimo para que el error no suba con el riesgo; la app une los puntos con rectas, así que el ± cambia poco a poco.

In [4]:
base = base_margin(train, blocks["routine"])
raw = tables["routine"]["years"].abs().round(2).tolist()
pd.DataFrame({"risk": base["risk"], "years, raw": raw, "years, isotonic": base["years"]}).round(3)

,risk,"years, raw","years, isotonic"
0,0.005,3.43,4.243
1,0.008,2.80,4.243
2,0.012,6.50,4.243
3,0.019,3.60,3.601
4,0.029,0.30,0.898
5,0.046,1.50,0.898
6,0.073,0.25,0.249
7,0.122,0.05,0.120
8,0.211,0.19,0.120
9,0.452,0.04,0.036


## 3. Por edad

Lo mismo por décadas de edad real: cuántos años habría que mover a cada grupo para que cuadre con sus muertes.

In [5]:
decades = pd.cut(train["age"], [17, 29, 39, 49, 59, 69, 80], labels=["18-29", "30-39", "40-49", "50-59", "60-69", "70-79"])
rows = {}
for level, (age, reference) in ages.items():
    for decade, part in train.groupby(decades, observed=True):
        members = np.asarray(decades == decade)
        years = years_to_observed(age[members], part["sex"], observed_risk(part), reference)
        rows[(decade, "deaths")] = deaths(part)
        rows[(decade, level)] = years
pd.Series(rows).unstack().round(1)

,deaths,extras,full,questionnaire,routine
18-29,81.0,6.3,7.3,6.5,5.2
30-39,89.0,0.9,1.0,0.2,-0.1
40-49,219.0,0.7,0.5,0.4,0.6
50-59,385.0,-0.1,-0.7,-0.5,0.2
60-69,831.0,-0.6,-1.1,-1.0,-0.6
70-79,1248.0,0.4,0.3,-0.2,-0.2


## 4. Conclusión

**El error base es una curva continua según el riesgo, la misma para los cuatro niveles:** la de `routine`, que tiene más muertes (4.008, frente a 1.478 de los niveles con cuestionario, que solo existe desde 2007).

| Riesgo a 10 años | Error base |
|---|---|
| Hasta un 1,2 % | 4,2 años |
| 1,9 % | 3,6 años |
| 2,9 % a 4,6 % | 0,9 años |
| 7 % | 0,25 años |
| 12 % o más | 0,1 años o menos |

Los décimos bajos son muy ruidosos (3,4 · 2,8 · 6,5 años, con solo 17 a 56 muertes cada uno): la isotónica los iguala en 4,2 en vez de dejar que un 6,5 suelto mande. Entre dos décimos, la app une con una recta, así que un cambio pequeño de un valor mueve poco el ±.

**Se descartó** dar dos tramos (por debajo y por encima de la mediana): la misma persona pasaba de ± 3 a ± 0,4 con un cambio mínimo de la glucosa. Y un error base por nivel: con cuestionario salía más ancho que sin él, por el ruido de tener menos muertes, no porque el modelo fuera peor.

**Sesgo en los jóvenes:** los de 18 a 29 años mueren más de lo que predice el modelo, que les quita de 5 a 7 años (solo 81 muertes). Por encima de 30, menos de 1,1 años en cualquier década. No se corrige: con tan pocas muertes sería ajustar al ruido.

**En la app:** `modeling/s7_margin.py` mide esta curva en cada reentreno y la guarda en cada modelo.